# Assignment 3 — Sampling Distributions and Estimation

**Course:** STA1

**Group members:** Bibek Paudel, Dora Buruncuk, Supendra Bogati

## Table of contents
1. [Part 1 — Sampling distributions and the CLT](#part1)
2. [Part 2 — Estimating mean sensor error](#part2)
3. [Part 3 — Estimating a false-rejection proportion](#part3)
4. [Part 4 — AI-use and conclusion](#part4)

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.proportion import proportion_confint

DATA = "data/"

access = pd.read_csv(DATA + "assignment03_access_control.csv")   # Part 3

print(f"access: {access.shape[0]} rows x {access.shape[1]} columns")

access: 240 rows x 2 columns


<a id="part3"></a>
# Part 3 — Estimating a false-rejection proportion

> The file `assignment03_access_control.csv` contains 240 independently staged legitimate login
> attempts. `false_reject` equals 1 when a legitimate attempt was rejected and 0 otherwise. Let $p$
> be the population false-rejection probability under the test conditions.

Let $Y_i = 1$ if legitimate attempt $i$ is falsely rejected and $Y_i = 0$ otherwise. The $Y_i$ are
independent Bernoulli trials with $P(Y_i = 1) = p$.

## a)

> Report $n$, the number of false rejections $x$, and $\hat p$. Explain why $\hat p$ is also the
> sample mean of a binary variable.

$$x = \sum_{i=1}^{n} y_i, \qquad \hat p = \frac{x}{n}$$

In [2]:
y = access["false_reject"]

n = len(y)
x = int(y.sum())
p_hat = x / n

print(f"n               = {n}")
print(f"x               = {x}")
print(f"p_hat = x / n   = {x}/{n} = {p_hat:.4f}")
print(f"mean of 0/1 col = {y.mean():.4f}")

n               = 240
x               = 20
p_hat = x / n   = 20/240 = 0.0833
mean of 0/1 col = 0.0833


**$\hat p = 20/240 \approx 0.0833$:** 8.3 % of the staged legitimate attempts were falsely rejected.

**Why $\hat p$ is a sample mean:** each $y_i$ is 0 or 1, so their sum counts the false rejections:

$$\bar y = \frac{1}{n}\sum_{i=1}^{n} y_i = \frac{x}{n} = \hat p$$

So the results for sample means apply to $\hat p$: $E[\hat p] = p$ and
$\text{SE}(\hat p) = \sqrt{p(1-p)/n}$.

## b)

> Construct and interpret a 95% Wilson interval for $p$ with statsmodels. State the population and
> parameter clearly.

For $z = z_{1-\alpha/2}$ the Wilson interval is

$$\frac{\hat p + \dfrac{z^2}{2n} \;\pm\; z\sqrt{\dfrac{\hat p(1-\hat p)}{n} + \dfrac{z^2}{4n^2}}}{1 + \dfrac{z^2}{n}}$$

with $z = z_{0.975} \approx 1.96$ for 95 % confidence.

In [3]:
wilson_lo, wilson_hi = proportion_confint(x, n, alpha=0.05, method="wilson")
print(f"95% Wilson interval = [{wilson_lo:.4f}, {wilson_hi:.4f}]")

95% Wilson interval = [0.0546, 0.1252]


**Population:** all legitimate login attempts under the same test conditions as the staged attempts.

**Parameter:** $p$, the probability that such an attempt is falsely rejected. It is fixed but
unknown; $\hat p$ estimates it.

**Interpretation:** the 95 % Wilson interval for $p$ is $[0.0546,\ 0.1252]$. If the study were
repeated many times with 240 new independent attempts, about 95 % of the intervals built this way
would contain the true $p$. This one interval either contains $p$ or it does not.

## c)

> Calculate the 95% Wald interval from its formula. Compare it with Wilson and explain why Wilson is
> the preferred default.

$$\widehat{\text{SE}}(\hat p) = \sqrt{\frac{\hat p(1-\hat p)}{n}}, \qquad
\text{ME} = z^{*}\,\widehat{\text{SE}}(\hat p), \qquad
\hat p \pm \text{ME}$$

In [4]:
z_95 = norm.ppf(0.975)
se_wald = np.sqrt(p_hat * (1 - p_hat) / n)
me_wald = z_95 * se_wald
wald_lo, wald_hi = p_hat - me_wald, p_hat + me_wald

print(f"p_hat              = {p_hat:.5f}")
print(f"z*                 = {z_95:.4f}")
print(f"estimated SE       = sqrt({p_hat:.5f} * {1 - p_hat:.5f} / {n}) = {se_wald:.5f}")
print(f"margin of error    = {z_95:.4f} * {se_wald:.5f} = {me_wald:.5f}")
print(f"95% Wald interval  = [{wald_lo:.4f}, {wald_hi:.4f}]")
print()

comparison = pd.DataFrame({
    "lower": [wald_lo, wilson_lo],
    "upper": [wald_hi, wilson_hi],
    "width": [wald_hi - wald_lo, wilson_hi - wilson_lo],
    "centre": [p_hat, (wilson_lo + wilson_hi) / 2],
}, index=["Wald", "Wilson"])
comparison.round(4)

p_hat              = 0.08333
z*                 = 1.9600
estimated SE       = sqrt(0.08333 * 0.91667 / 240) = 0.01784
margin of error    = 1.9600 * 0.01784 = 0.03497
95% Wald interval  = [0.0484, 0.1183]



,lower,upper,width,centre
Wald,0.0484,0.1183,0.0699,0.0833
Wilson,0.0546,0.1252,0.0706,0.0899


**Comparison:** both intervals are about equally wide (0.0699 and 0.0706). Wald is centred on
$\hat p = 0.0833$; Wilson is centred a little higher, at 0.0899. With 240 attempts they are similar.

**Why Wilson is the default:** Wald plugs $\hat p$ into the SE, so it can understate the
uncertainty, especially with small samples or $\hat p$ near 0 or 1. It can go below 0, and with
$x = 0$ it collapses to $[0,\ 0]$. Wilson stays inside $[0,\ 1]$ and does not collapse when $x = 0$.
Here $\hat p$ is small, so Wilson is the safer choice.

## d)

> Create a table of 90%, 95%, and 99% Wilson intervals and widths.

In [5]:
rows = []
for confidence in (0.90, 0.95, 0.99):
    lo, hi = proportion_confint(x, n, alpha=1 - confidence, method="wilson")
    rows.append({"confidence": f"{confidence:.0%}", "z*": norm.ppf((1 + confidence) / 2),
                 "lower": lo, "upper": hi, "width": hi - lo})

wilson_table = pd.DataFrame(rows).set_index("confidence")
wilson_table.round(4)

,z*,lower,upper,width
confidence,,,,
90%,1.6449,0.0584,0.1175,0.0591
95%,1.9600,0.0546,0.1252,0.0706
99%,2.5758,0.0478,0.1412,0.0934


**Interpretation:** higher confidence needs a larger $z^{*}$, so the interval gets wider:
from 0.059 at 90 % to 0.093 at 99 %.

## e)

> Estimate the sample size for a 95% margin of error no larger than 0.02 using first $p=0.5$ and
> then the observed $\hat p$. Round up and explain the difference.

Setting the margin of error $z^{*}\sqrt{p(1-p)/n}$ to at most $E$ and solving for $n$:

$$n \ge \frac{z^{*2}\, p(1-p)}{E^2}, \qquad E = 0.02,\quad z^{*} = z_{0.975}$$

In [6]:
E = 0.02

for label, p_plan in [("p = 0.5", 0.5), (f"p_hat = {p_hat:.4f}", p_hat)]:
    exact = z_95**2 * p_plan * (1 - p_plan) / E**2
    n_req = int(np.ceil(exact))                       # always round up
    print(f"{label:<15}: n >= {exact:.2f}  ->  n = {n_req}")
    for m in (n_req - 1, n_req):                      # margin at one fewer attempt and at n_req
        print(f"{'':17}n = {m}: margin = {z_95 * np.sqrt(p_plan * (1 - p_plan) / m):.6f}")

p = 0.5        : n >= 2400.91  ->  n = 2401
                 n = 2400: margin = 0.020004
                 n = 2401: margin = 0.020000
p_hat = 0.0833 : n >= 733.61  ->  n = 734
                 n = 733: margin = 0.020008
                 n = 734: margin = 0.019995


**Results:** $n = 2401$ with $p = 0.5$ and $n = 734$ with $\hat p = 0.0833$. We round up because one
attempt fewer gives a margin just above 0.02, as the check shows.

**Why they differ:** the margin depends on $p(1-p)$, which is largest at $p = 0.5$ (0.25) and much
smaller at $\hat p$ (about 0.076). So $p = 0.5$ is the worst case and works whatever the true $p$ is,
but needs over three times as many attempts. The $\hat p$ answer needs fewer attempts but relies on
an estimate of $p$.